In [0]:
%scala

import org.apache.spark.sql.functions._

// Read Bronze orders
val orders = spark.table("retail.bronze.orders")

// Schema
println("========== ORDERS SCHEMA ==========")
orders.printSchema()

// Total records
println(s"Total order records: ${orders.count()}")

// Display all records
orders.show(200, false)

========== ORDERS SCHEMA ==========
root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: string (nullable = true)
-- order_date: string (nullable = true)
-- order_status: string (nullable = true)
-- last_updated: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)
-- _source_file: string (nullable = true)
-- _batch_id: string (nullable = true)

Total order records: 161
+---------+-----------+----------+--------+----------+------------+-------------------+--------------------------+-------------------------------------------------------------+---------+
order_id |customer_id|product_id|quantity|order_date|order_status|last_updated |_ingestion_timestamp |_source_file |_batch_id|
+---------+-----------+----------+--------+----------+------------+-------------------+--------------------------+-------------------------------------------------------------+---------+
ORD000026|CUST020 |PRD004 |1 |2026-09-08|CANCELLED |2026-09-08 09:15:01|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000103|CUST018 |PRD012 |4 |2026-09-02|RETURNED |2026-09-03 15:33:29|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000089|CUST010 |PRD005 |2 |2026-09-20|CANCELLED |2026-09-20 17:35:45|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000032|CUST022 |PRD007 |5 |2026-11-14|DELIVERED |2026-11-14 22:01:34|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000138|CUST007 |PRD008 |4 |2026-09-23|RETURNED |2026-09-23 19:29:37|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000047|CUST038 |PRD010 |5 |2026-11-08|DELIVERED |2026-11-08 11:54:54|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000038|CUST017 |PRD011 |2 |2026-10-06|PLACED |2026-10-06 16:30:26|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000073|CUST013 |PRD019 |1 |NULL |DELIVERED |2026-10-18 09:35:44|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000095|CUST004 |PRD004 |3 |2026-11-05|PLACED |2026-11-05 08:30:00|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000065|CUST036 |PRD004 |3 |2026-10-05|DELIVERED |2026-10-05 11:21:09|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000078|CUST021 |PRD009 |1 |2026-11-17|CANCELLED |2026-11-17 15:16:31|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000043|CUST036 |PRD017 |5 |2026-10-16|DELIVERED |2026-10-16 15:40:29|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000036|CUST888 |PRD001 |5 |2026-09-29|CANCELLED |2026-09-29 20:08:11|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000091|CUST023 |PRD005 |4 |2026-09-18|DELIVERED |2026-09-18 12:34:44|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000117|CUST038 |PRD001 |3 |2026-10-11|PLACED |2026-10-11 12:49:27|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000018|CUST022 |PRD015 |-2 |2026-09-01|CANCELLED |2026-09-01 09:56:29|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000002|CUST032 |PRD005 |5 |2026-11-14|DELIVERED |2026-11-14 17:49:59|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000129|CUST012 |PRD006 |2 |2026-09-12|DELIVERED |2026-09-12 15:43:39|2026-11-20 09:38:44.709097|abfss://r

In [0]:
%scala
// Business columns used to compare order records
val orderColumns = Seq(
  "order_id",
  "customer_id",
  "product_id",
  "quantity",
  "order_date",
  "order_status",
  "last_updated"
)

// Analyze duplicate order IDs
val duplicateOrderAnalysis = orders
  .groupBy("order_id")
  .agg(
    count("*").alias("total_count"),
    countDistinct(
      struct(orderColumns.map(col): _*)
    ).alias("distinct_versions")
  )
  .filter(col("total_count") > 1)

// Exact duplicate order IDs
val exactDuplicateOrderIds = duplicateOrderAnalysis
  .filter(col("distinct_versions") === 1)

// Conflicting duplicate order IDs
val conflictingOrderIds = duplicateOrderAnalysis
  .filter(col("distinct_versions") > 1)

println("========== ORDER DUPLICATE ANALYSIS ==========")

println("All duplicate order IDs:")
duplicateOrderAnalysis.orderBy("order_id").show(false)

println("Exact duplicate order IDs:")
exactDuplicateOrderIds.show(false)

println("Conflicting duplicate order IDs:")
conflictingOrderIds.show(false)

========== ORDER DUPLICATE ANALYSIS ==========
All duplicate order IDs:
+---------+-----------+-----------------+
order_id |total_count|distinct_versions|
+---------+-----------+-----------------+
ORD000091|2 |1 |
ORD000092|2 |1 |
ORD000093|2 |1 |
ORD000094|2 |1 |
ORD000095|2 |1 |
ORD000101|2 |2 |
ORD000102|2 |2 |
ORD000103|2 |2 |
ORD000104|2 |2 |
ORD000105|2 |2 |
ORD000106|2 |2 |
+---------+-----------+-----------------+

Exact duplicate order IDs:
+---------+-----------+-----------------+
order_id |total_count|distinct_versions|
+---------+-----------+-----------------+
ORD000092|2 |1 |
ORD000091|2 |1 |
ORD000093|2 |1 |
ORD000094|2 |1 |
ORD000095|2 |1 |
+---------+-----------+-----------------+

Conflicting duplicate order IDs:
+---------+-----------+-----------------+
order_id |total_count|distinct_versions|
+---------+-----------+-----------------+
ORD000101|2 |2 |
ORD000106|2 |2 |
ORD000104|2 |2 |
ORD000105|2 |2 |
ORD000103|2 |2 |
ORD000102|2 |2 |
+---------+-----------+-----------------+

val orderColumns: Seq[String] = List(order_id, customer_id, product_id, quantity, order_date, order_status, last_updated)
val duplicateOrderAnalysis: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, total_count: bigint ... 1 more field]
val exactDuplicateOrderIds: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, total_count: bigint ... 1 more field]
val conflictingOrderIds: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, total_count: bigint ... 1 more field]

In [0]:
%scala
println("========== CONFLICTING ORDER RECORDS ==========")

val conflictingOrderRecords = orders
  .join(
    conflictingOrderIds.select("order_id"),
    Seq("order_id"),
    "inner"
  )
  .orderBy(
    col("order_id"),
    col("last_updated").desc
  )

conflictingOrderRecords.show(50, false)

========== CONFLICTING ORDER RECORDS ==========
+---------+-----------+----------+--------+----------+------------+-------------------+--------------------------+-------------------------------------------------------------+---------+
order_id |customer_id|product_id|quantity|order_date|order_status|last_updated |_ingestion_timestamp |_source_file |_batch_id|
+---------+-----------+----------+--------+----------+------------+-------------------+--------------------------+-------------------------------------------------------------+---------+
ORD000101|CUST028 |PRD007 |4 |2026-09-21|CANCELLED |2026-09-21 17:11:13|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000101|CUST028 |PRD007 |4 |2026-09-21|DELIVERED |2026-09-21 09:11:13|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000102|CUST018 |PRD010 |3 |2026-11-10|DELIVERED |2026-11-12 06:41:57|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000102|CUST018 |PRD010 |3 |2026-11-10|DELIVERED |2026-11-10 10:41:57|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000103|CUST018 |PRD012 |4 |2026-09-02|RETURNED |2026-09-03 15:33:29|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000103|CUST018 |PRD012 |4 |2026-09-02|DELIVERED |2026-09-02 15:33:29|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000104|CUST015 |PRD012 |4 |2026-09-14|SHIPPED |2026-09-17 00:09:35|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000104|CUST015 |PRD012 |4 |2026-09-14|DELIVERED |2026-09-14 20:09:35|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000105|CUST036 |PRD011 |4 |2026-11-17|DELIVERED |2026-11-18 01:37:03|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000105|CUST036 |PRD011 |3 |2026-11-17|CANCELLED |2026-11-17 18:37:03|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000106|CUST035 |PRD018 |5 |2026-09-11|CANCELLED |2026-09-13 23:29:22|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000106|CUST035 |PRD018 |5 |2026-09-11|DELIVERED |2026-09-11 15:29:22|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
+---------+-----------+----------+--------+----------+------------+-------------------+--------------------------+-------------------------------------------------------------+---------+

val conflictingOrderRecords: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 8 more fields]

In [0]:
%scala
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._

// Step 1: Remove exact duplicate records
val ordersWithoutExactDuplicates =
  orders.dropDuplicates(orderColumns)

// Step 2: Convert last_updated into timestamp
val ordersWithTimestamp =
  ordersWithoutExactDuplicates
    .withColumn(
      "last_updated_ts",
      to_timestamp(col("last_updated"), "yyyy-MM-dd HH:mm:ss")
    )

// Step 3: Define window
val orderWindow = Window
  .partitionBy("order_id")
  .orderBy(col("last_updated_ts").desc)

// Step 4: Rank each version
val rankedOrders = ordersWithTimestamp
  .withColumn(
    "version_rank",
    row_number().over(orderWindow)
  )

// Step 5: Keep latest version
val deduplicatedOrders = rankedOrders
  .filter(col("version_rank") === 1)
  .drop("version_rank", "last_updated_ts")

println("Original count: " + orders.count())
println("After exact deduplication: " + ordersWithoutExactDuplicates.count())
println("After version resolution: " + deduplicatedOrders.count())

deduplicatedOrders.orderBy("order_id").show(20, false)

Original count: 161
After exact deduplication: 156
After version resolution: 150
+---------+-----------+----------+--------+----------+------------+-------------------+--------------------------+-------------------------------------------------------------+---------+
order_id |customer_id|product_id|quantity|order_date|order_status|last_updated |_ingestion_timestamp |_source_file |_batch_id|
+---------+-----------+----------+--------+----------+------------+-------------------+--------------------------+-------------------------------------------------------------+---------+
ORD000001|CUST023 |PRD005 |1 |2026-09-06|DELIVERED |2026-09-06 20:34:17|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000002|CUST032 |PRD005 |5 |2026-11-14|DELIVERED |2026-11-14 17:49:59|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000003|CUST033 |PRD010 |1 |2026-10-31|DELIVERED |2026-10-31 17:08:38|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000004|CUST038 |5 |5 |2026-09-21|DELIVERED |2026-09-21 22:08:50|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000005|CUST013 |PRD013 |1 |2026-10-20|DELIVERED |2026-10-20 20:09:33|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000006|CUST023 |PRD022 |0 |2026-10-20|RETURNED |2026-10-20 22:04:03|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000007|CUST021 |PRD024 |3 |2026-09-04|DELIVERED |2026-09-04 08:32:30|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000008|CUST030 |PRD020 |5 |2026-10-12|DELIVERED |2026-10-12 22:07:31|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000009|NULL |PRD021 |3 |2026-09-20|DELIVERED |2026-09-20 17:33:13|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000010|CUST005 |PRD019 |4 |2026-11-16|DELIVERED |2026-11-16 11:43:45|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000011|CUST003 |PRD022 |5 |2026-09-26|DELIVERED |2026-09-26 18:58:11|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000012|CUST013 |PRD017 |1 |2026-10-28|DELIVERED |2026-10-28 21:48:41|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000013|CUST034 |PRD777 |2 |2026-11-07|CANCELLED |2026-11-07 11:50:56|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000014|CUST035 |PRD015 |1 |2026-09-24|DELIVERED |2026-09-24 14:17:27|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000015|CUST017 |PRD016 |4 |2026-10-10|RETURNED |2026-10-10 10:22:11|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000016|CUST021 |PRD018 |4 |2026-10-24|DELIVERED |2026-10-24 12:29:11|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000017|CUST018 |PRD004 |2 |2026-09-16|DELIVERED |2026-09-16 16:50:33|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000018|CUST022 |PRD015 |-2 |2026-09-01|CANCELLED |2026-09-01 09:56:29|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000019|CUST011 |PRD012 |5 |2026-09-24|CANCELLED |2026-09-24 08:22:54|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000020|CUST008 |PRD021 |3 |2026-11-13|SHIPPED |2026-11-13 20:34:13|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|

In [0]:

%scala

// ==========================================
// ORDER DATA NORMALIZATION
// ==========================================

val ordersNormalized = deduplicatedOrders

  // 1. Convert quantity from STRING to INT
  .withColumn(
    "quantity",
    expr("try_cast(quantity AS INT)")
  )

  // 2. Parse supported date formats safely
  .withColumn(
    "order_date",
    coalesce(
      expr("try_to_date(trim(order_date), 'yyyy-MM-dd')"),
      expr("try_to_date(trim(order_date), 'dd-MM-yyyy')")
    )
  )

  // 3. Standardize order status
  .withColumn(
    "order_status",
    upper(trim(col("order_status")))
  )

// ==========================================
// QUANTITY VALIDATION
// ==========================================

val invalidQuantityOrders = ordersNormalized.filter(
  col("quantity").isNull ||
  col("quantity") <= 0
)

// ==========================================
// DATE VALIDATION
// ==========================================

val invalidDateOrders = ordersNormalized.filter(
  col("order_date").isNull ||
  col("order_date") > lit("2026-11-20").cast("date")
)

// ==========================================
// ORDER STATUS VALIDATION
// ==========================================

val validStatuses = Seq(
  "PLACED",
  "SHIPPED",
  "DELIVERED",
  "CANCELLED",
  "RETURNED"
)

val invalidStatusOrders = ordersNormalized.filter(
  col("order_status").isNull ||
  !col("order_status").isin(validStatuses: _*)
)

// ==========================================
// VALIDATION RESULTS
// ==========================================

println("========== ORDER VALIDATION PROFILE ==========")

println("\nInvalid quantity records: " + invalidQuantityOrders.count())

invalidQuantityOrders
  .select("order_id", "quantity")
  .show(false)

println("\nInvalid date records: " + invalidDateOrders.count())

invalidDateOrders
  .select("order_id", "order_date")
  .show(false)

println("\nInvalid status records: " + invalidStatusOrders.count())

invalidStatusOrders
  .select("order_id", "order_status")
  .show(false)


========== ORDER VALIDATION PROFILE ==========

Invalid quantity records: 4
+---------+--------+
order_id |quantity|
+---------+--------+
ORD000006|0 |
ORD000018|-2 |
ORD000030|NULL |
ORD000042|NULL |
+---------+--------+


Invalid date records: 3
+---------+----------+
order_id |order_date|
+---------+----------+
ORD000071|NULL |
ORD000072|2026-12-25|
ORD000073|NULL |
+---------+----------+


Invalid status records: 3
+---------+------------+
order_id |order_status|
+---------+------------+
ORD000054|DLVRD |
ORD000057|UNKNOWN |
ORD000081|NULL |
+---------+------------+

val ordersNormalized: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 8 more fields]
val invalidQuantityOrders: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 8 more fields]
val invalidDateOrders: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 8 more fields]
val validStatuses: Seq[String] = List(PLACED, SHIPPED, DELIVERED, CANCELLED, RETURNED)
val invalidStatusOrders: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 8 more fields]

In [0]:

%scala

// ==========================================
// LOAD VALID SILVER DIMENSION REFERENCES
// ==========================================

val validCustomerIds = spark.table("retail.silver.customers")
  .select("customer_id")
  .distinct()

val validProductIds = spark.table("retail.silver.products")
  .select("product_id")
  .distinct()

// ==========================================
// CUSTOMER FOREIGN KEY VALIDATION
// ==========================================

// Missing customer IDs
val missingCustomerOrders = ordersNormalized.filter(
  col("customer_id").isNull ||
  trim(col("customer_id")) === ""
)

// Customer IDs that do not exist in Silver
val invalidCustomerOrders = ordersNormalized
  .filter(
    col("customer_id").isNotNull &&
    trim(col("customer_id")) =!= ""
  )
  .join(
    validCustomerIds,
    Seq("customer_id"),
    "left_anti"
  )

// ==========================================
// PRODUCT FOREIGN KEY VALIDATION
// ==========================================

// Missing product IDs
val missingProductOrders = ordersNormalized.filter(
  col("product_id").isNull ||
  trim(col("product_id")) === ""
)

// Product IDs that do not exist in Silver
val invalidProductOrders = ordersNormalized
  .filter(
    col("product_id").isNotNull &&
    trim(col("product_id")) =!= ""
  )
  .join(
    validProductIds,
    Seq("product_id"),
    "left_anti"
  )

// ==========================================
// DISPLAY RESULTS
// ==========================================

println("========== CUSTOMER FK VALIDATION ==========")

println("Missing customer IDs: " + missingCustomerOrders.count())
missingCustomerOrders.select("order_id", "customer_id").show(false)

println("Invalid customer references: " + invalidCustomerOrders.count())
invalidCustomerOrders.select("order_id", "customer_id").show(false)

println("\n========== PRODUCT FK VALIDATION ==========")

println("Missing product IDs: " + missingProductOrders.count())
missingProductOrders.select("order_id", "product_id").show(false)

println("Invalid product references: " + invalidProductOrders.count())
invalidProductOrders.select("order_id", "product_id").show(false)

========== CUSTOMER FK VALIDATION ==========
Missing customer IDs: 1
+---------+-----------+
order_id |customer_id|
+---------+-----------+
ORD000009|NULL |
+---------+-----------+

Invalid customer references: 29
+---------+-----------+
order_id |customer_id|
+---------+-----------+
ORD000005|CUST013 |
ORD000012|CUST013 |
ORD000013|CUST034 |
ORD000015|CUST017 |
ORD000020|CUST008 |
ORD000022|CUST017 |
ORD000023|CUST999 |
ORD000029|CUST008 |
ORD000031|CUST008 |
ORD000033|CUST008 |
ORD000036|CUST888 |
ORD000038|CUST017 |
ORD000046|CUST017 |
ORD000051|CUST013 |
ORD000061|CUST013 |
ORD000066|CUST026 |
ORD000071|CUST008 |
ORD000073|CUST013 |
ORD000075|CUST034 |
ORD000080|CUST026 |
+---------+-----------+
only showing top 20 rows

========== PRODUCT FK VALIDATION ==========
Missing product IDs: 1
+---------+----------+
order_id |product_id|
+---------+----------+
ORD000045|NULL |
+---------+----------+

Invalid product references: 33
+---------+----------+
order_id |product_id|
+---------+----------+
ORD000004|5 |
ORD000013|PRD777 |
ORD000015|PRD016 |
ORD000017|PRD004 |
ORD000022|PRD004 |
ORD000025|PRD009 |
ORD000026|PRD004 |
ORD000027|PRD011 |
ORD000038|PRD011 |
ORD000042|PRD011 |
ORD000049|PRD009 |
ORD000055|PRD004 |
ORD000061|PRD016 |
ORD000064|PRD009 |
ORD000065|PRD004 |
ORD000077|PRD011 |
ORD000078|PRD009 |
ORD000079|PRD009 |
ORD000084|PRD009 |
ORD000090|PRD009 |
+---------+----------+
only showing top 20 rows
val validCustomerIds: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [customer_id: string]
val validProductIds: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [product_id: string]
val missingCustomerOrders: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 8 more fields]
val invalidCustomerOrders: org.apache.spark.sql.DataFrame = [customer_id: string, order_id: string ... 8 more fields]
val missingProductOrders: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 8 more fields]
val invalidProductOrders: org.apache.spark.sql.DataFrame = [product_id: string, order_id: string ... 8 more fields]

In [0]:
%scala



// ==========================================
// PREPARE CLEAN REFERENCE DATA
// ==========================================

val customerReference = validCustomerIds
  .withColumnRenamed("customer_id", "_matched_customer_id")

val productReference = validProductIds
  .withColumnRenamed("product_id", "_matched_product_id")

// ==========================================
// JOIN ORDERS WITH VALID DIMENSIONS
// ==========================================

val ordersWithReferences = ordersNormalized
  .join(
    customerReference,
    col("customer_id") === col("_matched_customer_id"),
    "left"
  )
  .join(
    productReference,
    col("product_id") === col("_matched_product_id"),
    "left"
  )

// ==========================================
// ASSIGN REJECTION REASONS
// ==========================================

val ordersWithValidation = ordersWithReferences
  .withColumn(
    "rejection_reason",
    concat_ws(
      " | ",

      when(
        col("quantity").isNull || col("quantity") <= 0,
        lit("INVALID_QUANTITY")
      ),

      when(
        col("order_date").isNull,
        lit("INVALID_ORDER_DATE")
      ),

      when(
        col("order_date") > lit("2026-11-20").cast("date"),
        lit("FUTURE_ORDER_DATE")
      ),

      when(
        col("order_status").isNull ||
        !col("order_status").isin(validStatuses: _*),
        lit("INVALID_ORDER_STATUS")
      ),

      when(
        col("customer_id").isNull ||
        trim(col("customer_id")) === "",
        lit("MISSING_CUSTOMER_ID")
      ),

      when(
        col("customer_id").isNotNull &&
        trim(col("customer_id")) =!= "" &&
        col("_matched_customer_id").isNull,
        lit("INVALID_CUSTOMER_REFERENCE")
      ),

      when(
        col("product_id").isNull ||
        trim(col("product_id")) === "",
        lit("MISSING_PRODUCT_ID")
      ),

      when(
        col("product_id").isNotNull &&
        trim(col("product_id")) =!= "" &&
        col("_matched_product_id").isNull,
        lit("INVALID_PRODUCT_REFERENCE")
      )
    )
  )
  .drop("_matched_customer_id", "_matched_product_id")

// ==========================================
// SPLIT VALID AND REJECTED ORDERS
// ==========================================

val validOrders = ordersWithValidation
  .filter(col("rejection_reason") === "" || col("rejection_reason").isNull)
  .drop("rejection_reason")

val rejectedOrders = ordersWithValidation
  .filter(col("rejection_reason") =!= "")

// ==========================================
// DISPLAY VALIDATION SUMMARY
// ==========================================

println("========== FINAL ORDER VALIDATION SUMMARY ==========")

println("Total deduplicated orders: " + deduplicatedOrders.count())

println("Valid orders: " + validOrders.count())

println("Rejected orders: " + rejectedOrders.count())

println("Total after validation: " +
  (validOrders.count() + rejectedOrders.count()))

println("\n========== REJECTION REASON DETAILS ==========")

rejectedOrders
  .select("order_id", "rejection_reason")
  .orderBy("order_id")
  .show(200, false)

println("\n========== VALID ORDER SAMPLE ==========")

validOrders.show(10, false)


========== FINAL ORDER VALIDATION SUMMARY ==========
Total deduplicated orders: 150
Valid orders: 87
Rejected orders: 63
Total after validation: 150

========== REJECTION REASON DETAILS ==========
+---------+------------------------------------------------------+
order_id |rejection_reason |
+---------+------------------------------------------------------+
ORD000004|INVALID_PRODUCT_REFERENCE |
ORD000005|INVALID_CUSTOMER_REFERENCE |
ORD000006|INVALID_QUANTITY |
ORD000009|MISSING_CUSTOMER_ID |
ORD000012|INVALID_CUSTOMER_REFERENCE |
ORD000013|INVALID_CUSTOMER_REFERENCE | INVALID_PRODUCT_REFERENCE|
ORD000015|INVALID_CUSTOMER_REFERENCE | INVALID_PRODUCT_REFERENCE|
ORD000017|INVALID_PRODUCT_REFERENCE |
ORD000018|INVALID_QUANTITY |
ORD000020|INVALID_CUSTOMER_REFERENCE |
ORD000022|INVALID_CUSTOMER_REFERENCE | INVALID_PRODUCT_REFERENCE|
ORD000023|INVALID_CUSTOMER_REFERENCE |
ORD000025|INVALID_PRODUCT_REFERENCE |
ORD000026|INVALID_PRODUCT_REFERENCE |
ORD000027|INVALID_PRODUCT_REFERENCE |
ORD000029|INVALID_CUSTOMER_REFERENCE |
ORD000030|INVALID_QUANTITY |
ORD000031|INVALID_CUSTOMER_REFERENCE |
ORD000033|INVALID_CUSTOMER_REFERENCE |
ORD000036|INVALID_CUSTOMER_REFERENCE |
ORD000038|INVALID_CUSTOMER_REFERENCE | INVALID_PRODUCT_REFERENCE|
ORD000042|INVALID_QUANTITY | INVALID_PRODUCT_REFERENCE |
ORD000045|MISSING_PRODUCT_ID |
ORD000046|INVALID_CUSTOMER_REFERENCE |
ORD000049|INVALID_PRODUCT_REFERENCE |
ORD000051|INVALID_CUSTOMER_REFERENCE |
ORD000054|INVALID_ORDER_STATUS |
ORD000055|INVALID_PRODUCT_REFERENCE |
ORD000057|INVALID_ORDER_STATUS |
ORD000061|INVALID_CUSTOMER_REFERENCE | INVALID_PRODUCT_REFERENCE|
ORD000064|INVALID_PRODUCT_REFERENCE |
ORD000065|INVALID_PRODUCT_REFERENCE |
ORD000066|INVALID_CUSTOMER_REFERENCE |
ORD000071|INVALID_ORDER_DATE | INVALID_CUSTOMER_REFERENCE |
ORD000072|FUTURE_ORDER_DATE |
ORD000073|INVALID_ORDER_DATE | INVALID_CUSTOMER_REFERENCE |
ORD000075|INVALID_CUSTOMER_REFERENCE |
ORD000077|INVALID_PRODUCT_REFERENCE |
ORD000078|INVALID_PRODUCT_REFERENCE |
ORD000079|INVALID_PRODUCT_REFERENCE |
ORD000080|INVALID_CUSTOMER_REFERENCE |
ORD000081|INVALID_ORDER_STATUS |
ORD000084|INVALID_PRODUCT_REFERENCE |
ORD000086|INVALID_CUSTOMER_REFERENCE |
ORD000090|INVALID_PRODUCT_REFERENCE |
ORD000095|INVALID_PRODUCT_REFERENCE |
ORD000100|INVALID_CUSTOMER_REFERENCE | INVALID_PRODUCT_REFERENCE|
ORD000105|INVALID_PRODUCT_REFERENCE |
ORD000108|INVALID_PRODUCT_REFERENCE |
ORD000110|INVALID_CUSTOMER_REFERENCE |
ORD000111|INVALID_CUSTOMER_REFERENCE | INVALID_PRODUCT_REFERENCE|
ORD000114|INVALID_CUSTOMER_REFERENCE |
ORD000119|INVALID_CUSTOMER_REFERENCE | INVALID_PRODUCT_REFERENCE|
ORD000120|INVALID_PRODUCT_REFERENCE |
ORD000121|INVALID_CUSTOMER_REFERENCE |
ORD000124|INVALID_PRODUCT_REFERENCE |
ORD000125|INVALID_PRODUCT_REFERENCE |
ORD000126|INVALID_CUSTOMER_REFERENCE |
ORD000134|INVALID_PRODUCT_REFERENCE |
ORD000139|INVALID_PRODUCT_REFERENCE |
ORD000142|INVALID_PRODUCT_REFERENCE |
ORD000143|INVALID_PRODUCT_REFERENCE |
ORD000148|INVALID_CUSTOMER_REFERENCE |
+---------+------------------------------------------------------+


========== VALID ORDER SAMPLE ==========
+---------+-----------+----------+--------+----------+------------+-------------------+--------------------------+-------------------------------------------------------------+---------+
order_id |customer_id|product_id|quantity|order_date|order_status|last_updated |_ingestion_timestamp |_source_file |_batch_id|
+---------+-----------+----------+--------+----------+------------+-------------------+--------------------------+-------------------------------------------------------------+---------+
ORD000001|CUST023 |PRD005 |1 |2026-09-06|DELIVERED |2026-09-06 20:34:17|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000002|CUST032 |PRD005 |5 |2026-11-14|DELIVERED |2026-11-14 17:49:59|2026-11-20 09:38:44.709097|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders.csv|BATCH_001|
ORD000003|CUST033 |PRD010 |1 |2026-10-31|DELIVER

In [0]:
%scala

println("========== SILVER ORDER QUALITY CHECKS ==========")

// 1. Check duplicate order IDs in valid orders
val duplicateValidOrders = validOrders
  .groupBy("order_id")
  .count()
  .filter(col("count") > 1)

// 2. Check invalid customer references
val orphanCustomers = validOrders
  .join(validCustomerIds, Seq("customer_id"), "left_anti")

// 3. Check invalid product references
val orphanProducts = validOrders
  .join(validProductIds, Seq("product_id"), "left_anti")

// 4. Check invalid quantities
val invalidValidQuantities = validOrders.filter(
  col("quantity").isNull || col("quantity") <= 0
)

// 5. Check invalid dates
val invalidValidDates = validOrders.filter(
  col("order_date").isNull ||
  col("order_date") > lit("2026-11-20").cast("date")
)

// 6. Check invalid statuses
val invalidValidStatuses = validOrders.filter(
  col("order_status").isNull ||
  !col("order_status").isin(validStatuses: _*)
)

println("Duplicate order IDs: " + duplicateValidOrders.count())
println("Orphan customer references: " + orphanCustomers.count())
println("Orphan product references: " + orphanProducts.count())
println("Invalid quantities: " + invalidValidQuantities.count())
println("Invalid dates: " + invalidValidDates.count())
println("Invalid statuses: " + invalidValidStatuses.count())

========== SILVER ORDER QUALITY CHECKS ==========
Duplicate order IDs: 0
Orphan customer references: 0
Orphan product references: 0
Invalid quantities: 0
Invalid dates: 0
Invalid statuses: 0
val duplicateValidOrders: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, count: bigint]
val orphanCustomers: org.apache.spark.sql.DataFrame = [customer_id: string, order_id: string ... 8 more fields]
val orphanProducts: org.apache.spark.sql.DataFrame = [product_id: string, order_id: string ... 8 more fields]
val invalidValidQuantities: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 8 more fields]
val invalidValidDates: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 8 more fields]
val invalidValidStatuses: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 8 more fields]

In [0]:

%scala

// ==========================================
// SAVE VALID ORDERS
// ==========================================

validOrders.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable("retail.silver.orders")

// ==========================================
// SAVE REJECTED ORDERS
// ==========================================

rejectedOrders.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable("retail.silver.rejected_orders")

println("Silver order tables created successfully!")

// ==========================================
// VERIFY TABLE COUNTS
// ==========================================

println("Valid orders: " +
  spark.table("retail.silver.orders").count())

println("Rejected orders: " +
  spark.table("retail.silver.rejected_orders").count())


Silver order tables created successfully!
Valid orders: 87
Rejected orders: 63